In [1]:
from sodapy import Socrata
import pandas as pd
import json,csv
import os,inspect,sys
from chlorophyll import CodeView
from tkinter import Tk
import tkinter as tk
# from tkinter import ttk,Button,Label
from datetime  import datetime
#import PySimpleGUI as sg
from difflib import SequenceMatcher
import pygments.lexers
import pytz
import glob
from croniter import croniter
from cron_descriptor import get_description, ExpressionDescriptor
os.environ["BROWSER"] = "google-chrome"
import webbrowser
browsers = webbrowser._tryorder
print("Browsers detected:", browsers)
today = datetime.today()
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
cim_url_query = 'data.colorado.gov'
datasets = None
tody = datetime.today()
#today=f"{tody.year}-{tody.month:02d}-{tody.day:02d}"    
from google.oauth2 import service_account
from googleapiclient.discovery import build
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
# import seaborn as sns,matplotlib.pyplot as plt
# import matplotlib.dates as mdates

bic_etl_home = os.getenv('bic_etl_home')

import requests
from requests.auth import HTTPBasicAuth


cimDatasets = {}
bicHome = "/home/joe/bic_etl"
allDatasets=[]

allDats={}
with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()
    for dataset in datasets:
        allDatasets.append(dataset)
        title=dataset["resource"]["name"]
        allDats[title]=dataset
        if dataset['owner']['display_name'] == 'Colorado Information Marketplace':
            title=dataset["resource"]["name"]
            cimDatasets[title]=dataset

print("DONE")

Browsers detected: None


DONE


In [ ]:
for title in sorted(allDats.keys()):
    print(title)

In [ ]:
for title in sorted(cimDatasets.keys()):
    print(title)

In [ ]:
sorted(cimDatasets.keys())

In [ ]:
cimDatasets["Housing Inventory: New Listing Count in Colorado"]["resource"]

In [18]:
def find_files_recursive(directory):
    # This function finds all files recursively in a given directory
    return glob.glob(f'{directory}/**/run_etl.json', recursive=True)

def getGroups(directory_path='/home/joe/bic_etl'): 
## Get all run_etl.json files
    files = find_files_recursive(directory_path)
    skip_groups = ["catalog","general/example"]
    datasets={}
    noEx=0
    groups={}
    for file in files:
        ll=len(file)
        group=file[18:-13]
        fin=open(file)
        infos = json.load(fin)
        fin.close()
        title=""
        extract=""
        for info in infos: 
            if 'title' in info:
                title=info['title']
                if group not in skip_groups:
                    if group not in groups:
                        groups[group]=[]
                    groups[group].append(title)
    return groups,files

groups,files=getGroups()

In [5]:
def getCronInfo(cron_file_path):
    fin=open(cron_file_path)
    lines=fin.readlines()
    crons={}
    for line in lines:
        if line[0:1] != "#" and line[0:1] != " " and len(line) > 2:
            line=line.strip()
            spl=line.split()
            crn=' '.join(spl[:5])
            rest = ' '.join(spl[5:])
            
            crons[rest]=crn
    return crons

def decodeCron(cron_file):
    hist={}
    cronGroups={}
    crons=getCronInfo(cron_file)
    for inf,crn in crons.items():
    
        desc = get_description(crn)
        spl=inf.split()
        lang=spl[0]
        prg=spl[1]
        indxP=inf.find("-p")
        if indxP > -1:
           tmp=inf[indxP+2:].lstrip()
           end=tmp.find(" ")
           grp=tmp[:end]
        else:
            grp=""
    
        indxT=inf.find("-t")
        if indxT > -1:
           tmp=inf[indxT+2:].lstrip()
           start = tmp.find('"')
           end=tmp[start+1:].find('"')
           titl=tmp[start+1:end+1]
        else:
            titl="ALL"
        if grp not in cronGroups:
            cronGroups[grp]={}
        cronGroups[grp][titl]=desc
   #     print(grp,titl)     
        if lang not in hist:
            hist[lang]={}
        if prg not in hist[lang]:
            hist[lang][prg]=0
        hist[lang][prg]+=1
    return cronGroups

## Combine Dataset ETL Info and CRON Info 
# def addCronInfo(row):
#     title=row['Title']
#     group=row['Group']
# #    print(group,title)
#     if group in cronGroups:
#         dct=cronGroups[group]
#         if title in dct:
#             cron=cronGroups[group][title]
#         elif "ALL" in dct:
#             cron=cronGroups[group]["ALL"]
#         else:
#             cron="Unknown"   
            
#     else:
#         cron="No Group Found"
#     return cron

cron_file = '/home/joe/bic_etl/general/cron/cron_file'    
cronGroups=decodeCron(cron_file)

In [16]:
cronGroups.keys()

dict_keys(['', 'bls/sm', 'boulder', 'catalog', 'cdos/business/nonprofit', 'cdos/business/business', 'cdos/health', 'cdos/lobbyist', 'cdos/government', 'cdos/business/ucc', 'cdot/transportation_road_attributes', 'cdot/transportation_infrastructure', 'cdot/natural_resources', 'cdot/tops', 'dola/boundaries', 'dola/special_districts', 'dola/demographics', 'cdor/revenue_marijuana', 'cdor/retail_reports', 'cdor/regulations_liquor', 'ceo/useia', 'fred'])

In [15]:
for grp in groups.keys():
    if grp in cronGroups:
        print(grp,cronGroups[grp])

fred {'ALL': 'At 03:30 AM, on day 7 of the month'}
cdos/health {'ALL': 'At 04:00 AM, only on Tuesday'}
cdos/government {'ALL': 'At 04:30 AM, only on Tuesday'}
cdos/business/ucc {'ALL': 'At 03:00 AM'}
cdos/business/business {'Business Entities in Colorado': 'At 05:10 AM', 'Business Entity Transaction History': 'At 08:00 AM', 'Master List in Colorado': 'At 05:30 AM, on day 4 of the month', 'Trade Names for Businesses in Colorado': 'At 04:00 AM', 'Trademarks for Businesses in Colorado': 'At 04:00 AM'}
cdos/business/nonprofit {'ALL': 'At 06:05 AM, only on Friday', 'Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado': 'At 05:05 AM'}
cdos/lobbyist {'ALL': 'At 04:15 AM'}
boulder {'Septic Systems in Boulder County Colorado': 'At 03:10 AM'}
cdot/transportation_road_attributes {'ALL': 'At 04:10 AM, on day 4 of the month'}
cdot/tops {'ALL': 'At 08:45 AM, only on Tuesday'}
cdot/natural_resources {'ALL': 'At 04:50

In [17]:
groups 

{'fred': ['Housing Inventory: New Listing Count in Colorado'],
 'bls': ['Average Weekly & Hourly Earnings and Work Hours Across Major Industry Sectors in Colorado',
  'Employment Counts Across Major Industry Sectors in Colorado',
  'Monthly Consumer Price Index Data for Denver-Aurora-Lakewood'],
 'cdos/health': ['Durable Medical Equipment Suppliers in Colorado'],
 'cdos/government': ['Current Notaries in Colorado'],
 'cdos/business/ucc': ['Uniform Commercial Code (UCC) Collateral Information in Colorado',
  'Uniform Commercial Code (UCC) Debtor Information in Colorado',
  'Uniform Commercial Code (UCC) Filing Information in Colorado',
  'Secured Party Information in Colorado'],
 'cdos/business/business': ['Business Entities in Colorado',
  'Business Entity Transaction History',
  'Trademarks for Businesses in Colorado',
  'Trade Names for Businesses in Colorado',
  'Master List in Colorado'],
 'cdos/business/nonprofit': ['Campaign Reports for Solicitation Notices to Charities in Colora

In [19]:
files

['/home/joe/bic_etl/catalog/run_etl.json',
 '/home/joe/bic_etl/fred/run_etl.json',
 '/home/joe/bic_etl/bls/run_etl.json',
 '/home/joe/bic_etl/cdos/health/run_etl.json',
 '/home/joe/bic_etl/cdos/government/run_etl.json',
 '/home/joe/bic_etl/cdos/business/ucc/run_etl.json',
 '/home/joe/bic_etl/cdos/business/business/run_etl.json',
 '/home/joe/bic_etl/cdos/business/nonprofit/run_etl.json',
 '/home/joe/bic_etl/cdos/lobbyist/run_etl.json',
 '/home/joe/bic_etl/cdhe/run_etl.json',
 '/home/joe/bic_etl/dpa/run_etl.json',
 '/home/joe/bic_etl/tchd/run_etl.json',
 '/home/joe/bic_etl/boulder/run_etl.json',
 '/home/joe/bic_etl/bea/run_etl.json',
 '/home/joe/bic_etl/denver/run_etl.json',
 '/home/joe/bic_etl/cdot/transportation_road_attributes/run_etl.json',
 '/home/joe/bic_etl/cdot/tops/run_etl.json',
 '/home/joe/bic_etl/cdot/natural_resources/run_etl.json',
 '/home/joe/bic_etl/cdot/transportation_infrastructure/run_etl.json',
 '/home/joe/bic_etl/irs/run_etl.json',
 '/home/joe/bic_etl/dola/special_di

In [22]:
for file in files:
    spl=file.split("/")
    nn=spl.index("bic_etl")
    print(nn,"/".join(spl[nn+1:]))

3 catalog/run_etl.json
3 fred/run_etl.json
3 bls/run_etl.json
3 cdos/health/run_etl.json
3 cdos/government/run_etl.json
3 cdos/business/ucc/run_etl.json
3 cdos/business/business/run_etl.json
3 cdos/business/nonprofit/run_etl.json
3 cdos/lobbyist/run_etl.json
3 cdhe/run_etl.json
3 dpa/run_etl.json
3 tchd/run_etl.json
3 boulder/run_etl.json
3 bea/run_etl.json
3 denver/run_etl.json
3 cdot/transportation_road_attributes/run_etl.json
3 cdot/tops/run_etl.json
3 cdot/natural_resources/run_etl.json
3 cdot/transportation_infrastructure/run_etl.json
3 irs/run_etl.json
3 dola/special_districts/run_etl.json
3 dola/boundaries/run_etl.json
3 dola/demographics/run_etl.json
3 cdor/revenue_marijuana/run_etl.json
3 cdor/retail_reports/run_etl.json
3 cdor/regulations_liquor/run_etl.json
3 ceo/useia/run_etl.json
3 general/example/run_etl.json
3 dora/regulations/run_etl.json
